# 04 Numerical Experiments

This notebook turns the chapter into measurable experiments. The goal is not to
produce isolated plots, but to connect algorithm choices to visible numerical
behavior.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    NaturalCubicSpline,
    chebyshev_nodes,
    lagrange_interpolate,
    piecewise_linear_interpolate,
)


## Experiment 1: equal nodes versus Chebyshev nodes

The Runge function shows that adding more equally spaced nodes can increase the
maximum error. Chebyshev nodes usually control this effect much better.


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 1200)
y_true = runge(x_plot)
node_counts = np.arange(5, 32, 2)
errors_equal = []
errors_cheb = []

for n in node_counts:
    x_equal = np.linspace(-1.0, 1.0, n)
    x_cheb = chebyshev_nodes(-1.0, 1.0, n)
    y_equal = lagrange_interpolate(x_equal, runge(x_equal), x_plot)
    y_cheb = lagrange_interpolate(x_cheb, runge(x_cheb), x_plot)
    errors_equal.append(np.max(np.abs(y_equal - y_true)))
    errors_cheb.append(np.max(np.abs(y_cheb - y_true)))

plt.figure(figsize=(8, 4.5))
plt.semilogy(node_counts, errors_equal, "o-", label="Equal nodes")
plt.semilogy(node_counts, errors_cheb, "o-", label="Chebyshev nodes")
plt.xlabel("number of nodes")
plt.ylabel("max error")
plt.title("Runge experiment: node placement controls error")
plt.legend()
plt.tight_layout()
plt.show()


## Experiment 2: global polynomial, piecewise linear, and spline

For moderate smooth data, splines often provide a better compromise than either
rough piecewise linear interpolation or a high-degree global polynomial.


In [ ]:
def target(x):
    return np.sin(2.5 * x) + 0.2 * x

x_data = np.linspace(0.0, 3.0, 9)
y_data = target(x_data)
x_eval = np.linspace(0.0, 3.0, 600)
y_exact = target(x_eval)

methods = {
    "global polynomial": lagrange_interpolate(x_data, y_data, x_eval),
    "piecewise linear": piecewise_linear_interpolate(x_data, y_data, x_eval),
    "natural cubic spline": NaturalCubicSpline.fit(x_data, y_data)(x_eval),
}

for name, values in methods.items():
    print(f"{name:22s} max error = {np.max(np.abs(values - y_exact)):10.3e}")

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, y_exact, color="black", label="target")
for name, values in methods.items():
    plt.plot(x_eval, values, label=name)
plt.scatter(x_data, y_data, color="black", s=18, zorder=3)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Method comparison on smooth data")
plt.legend()
plt.tight_layout()
plt.show()


## Experiment 3: interpolation under noise

Interpolation is exact at the data points. With noisy observations, that exact
constraint can become a disadvantage.


In [ ]:
rng = np.random.default_rng(7)
x_noisy = np.linspace(-1.0, 1.0, 13)
y_noisy = runge(x_noisy) + 0.04 * rng.normal(size=x_noisy.size)

poly_noisy = lagrange_interpolate(x_noisy, y_noisy, x_plot)
linear_noisy = piecewise_linear_interpolate(x_noisy, y_noisy, x_plot)
spline_noisy = NaturalCubicSpline.fit(x_noisy, y_noisy)(x_plot)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, y_true, color="black", label="true Runge function")
plt.plot(x_plot, poly_noisy, label="global polynomial through noisy data")
plt.plot(x_plot, linear_noisy, label="piecewise linear")
plt.plot(x_plot, spline_noisy, label="natural cubic spline")
plt.scatter(x_noisy, y_noisy, color="black", s=20, zorder=3, label="noisy data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Exact interpolation can preserve noise")
plt.legend()
plt.tight_layout()
plt.show()


## Experiment summary

* Equal nodes can fail on the Runge function as degree increases.
* Chebyshev nodes make global polynomial interpolation much more reliable for this test.
* Piecewise and spline methods are often more robust because they are local.
* Noisy data should raise a modeling question: do we need interpolation, or fitting?
